# MCP Classification Against Custom O*NET Taxonomy

This notebook provides a thin orchestration layer for classifying MCP servers against a custom O*NET occupational task hierarchy.

## Pipeline Overview

The classification runs through a three-stage LLM pipeline:

1. **Stage 1**: Screen for occupational relevance + select one of 11 high-level parent tasks
2. **Stage 2**: Select one mid-level task + assign workflow automation score (1-10)
3. **Stage 3**: Select O*NET task(s) + assign deployability score(s) (1-10)

## Module Structure

All functionality is in the `lib/` package:
- `config.py` - Model configs, API settings, constants
- `taxonomy.py` - O*NET hierarchy data structures
- `stages.py` - Prompt templates and parsing
- `llm_clients.py` - LLM call functions (sync + async, all providers)
- `pipeline.py` - Classification pipeline
- `multi_model.py` - Multi-model comparison
- `analysis.py` - Agreement and determinism analysis
- `utils.py` - Save/load and formatting helpers

See `CLAUDE.md` for full documentation.

---

## 0 Load Helpers

In [124]:
# Auto-reload modules on change (useful during development)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [125]:
# Import the library
from lib import (
    # Data loading
    load_taxonomy, load_mcp_data,
    
    # Configuration
    check_api_keys, MODELS, MODEL_NAME, MODELS_TO_RUN,
    OUTPUT_VERSION, MULTI_MODEL_VERSION,
    
    # Single-model pipeline
    run_classification,
    
    # Multi-model comparison
    run_multi_model_classification,
    prepare_multi_model_sample,
    build_long_model_comparison,
    create_combined_comparison,
    load_multi_model_results,
    
    # Analysis
    analyze_model_agreement,
    compare_model_runs,
    
    # Utilities
    save_results,
    print_result_summary,
    view_result,
    display_results_table,
    generate_prompt_1,
    generate_prompt_2,
    generate_prompt_3,
)

print("Library imported successfully.")

Library imported successfully.


---

## 1. Setup and Configuration

In [ ]:
# =============================================================================
# VERSION CONFIGURATION
# =============================================================================
# Change these version numbers to tag your output files.
# These override the defaults from lib/config.py.

# Version for single-model classification runs (save_results)
SINGLE_MODEL_VERSION = "v5"

# Version for multi-model comparison runs (run_multi_model_classification)
MULTI_MODEL_RUN_VERSION = "v2"

print(f"Single-model version: {SINGLE_MODEL_VERSION}")
print(f"Multi-model version:  {MULTI_MODEL_RUN_VERSION}")

In [127]:
# Check API key status
check_api_keys()

API Key Status:
  OPENAI: SET
  ANTHROPIC: SET
  XAI: SET
  GOOGLE: SET


In [128]:
# View available models and default model
print(f"Default model: {MODEL_NAME}")
print(f"\nAvailable models ({len(MODELS)}):")
for key, config in MODELS.items():
    print(f"  - {key}: {config['display_name']} ({config['provider']})")

Default model: gpt-4.1

Available models (8):
  - claude-4.5-haiku: Claude 4.5 Haiku (anthropic)
  - claude-sonnet-4: Claude Sonnet 4 (anthropic)
  - claude-sonnet-4.5: Claude Sonnet 4.5 (anthropic)
  - grok-3: Grok 3 (xai)
  - gpt-4.1: GPT-4.1 (openai)
  - gpt-5: GPT-5 (openai)
  - gpt-5-mini: GPT-5 Mini (openai)
  - gemini-2.5-pro: Gemini 2.5 Pro (google)


In [146]:
# Load data
taxonomy = load_taxonomy()
mcp_df = load_mcp_data()

# Show taxonomy summary
taxonomy.summary()

Loaded taxonomy with 17538 rows from c:\Users\teddy\Downloads\OAIPR\Technical\aei_data_analysis\mcp\custom_tax_class\final_onet_taxonomy.csv
Columns: ['high_level_task', 'mid_level_task', 'original_onet_task']
Loaded 8953 MCP servers from c:\Users\teddy\Downloads\OAIPR\Technical\aei_data_analysis\mcp\custom_tax_class\mcp_desc_all_jan_22_cleaned.csv
Columns: ['title', 'url', 'key_features', 'uploaded_clean', 'text_for_llm', 'text_for_llm_2', 'len_text']
Taxonomy Summary:
  High-level tasks: 11
  Mid-level tasks: 426
  O*NET tasks: 17538

High-level task previews:
  1. Plan, perform, and coordinate hands-on service and operational activities across facilities, clients...
  2. Plan, install, set up, start, operate, monitor, inspect, test, calibrate, adjust, troubleshoot, and ...
  3. Plan, deliver, and coordinate comprehensive patient care across the continuum by assessing and diagn...
  4. Plan, direct, and administer organizational financial, legal/regulatory, and records/IT operations 

In [150]:
print(taxonomy["mid_level_task"])

TypeError: 'Taxonomy' object is not subscriptable

---

## 2. Single-Model Classification

Run the 3-stage pipeline on a sample of MCPs using a single model.

In [130]:
# Configuration for sample run
SAMPLE_TITLES = {
    # "mcp-server-rubygems",
    # "iRacing",
    # "Gitee MCP Server",
    # "mxHERO Mail2Cloud MCP",
    # "Filament MCP Server - Laravel Loop",
    "SumoLogic MCP Server",
    # "SQL Server MCP Server",
    "Vapi MCP for Cursor",
    "Shopify MCP Server for Claude",
    # "DART-mcp-server"
}

# Select sample
sample_df = mcp_df[mcp_df['title'].isin(SAMPLE_TITLES)].copy().reset_index(drop=True)
print(f"Sample: {len(sample_df)} MCPs")
print(f"Titles: {list(sample_df['title'])}")

Sample: 3 MCPs
Titles: ['SumoLogic MCP Server', 'Vapi MCP for Cursor', 'Shopify MCP Server for Claude']


In [131]:
# Run classification
results_df = run_classification(
    df=sample_df,
    taxonomy=taxonomy,
    model_name="gpt-4.1"  # or any model from lib.MODELS
)

print(f"\nResults shape: {results_df.shape}")

Starting classification of 3 MCPs...
  Model: gpt-4.1
  Max concurrent requests: 50
  Completed 3/3 (0.5/sec)

Classification complete!
  Total time: 6.3 seconds
  Average rate: 0.5 MCPs/sec

Results shape: (3, 15)


In [132]:
# Review results
print_result_summary(results_df)

Classification Results Summary

Occupational Relevance:
occupational_relevance
Yes    3
Name: count, dtype: int64

Relevant MCPs: 3 / 3

High-Level Task Distribution (top 5):
  1: Plan, direct, and administer organizational financial, legal/regulatory, and rec...
  1: Plan, direct, and administer organizational financial, legal/regulatory, and rec...
  1: Plan, direct, and administer organizational financial, legal/regulatory, and rec...

Workflow Automation Scores:
  Count: 3
  Mean: 6.67
  Median: 7.0

Errors: 0 / 3


In [133]:
# View results table
display_results_table(results_df)

,title,occupational_relevance,high_level_task,mid_level_task,workflow_automation,onet_tasks,deployability,error
0,SumoLogic MCP Server,Yes,"Plan, direct, and administer organizational fi...","Prepare and deliver reports, presentations, an...",6,"Compile reports, charts, or graphs that descri...",3;3;3;3;3;3;3,None
1,Vapi MCP for Cursor,Yes,"Plan, direct, and administer organizational fi...","Process orders, billing, and payments by calcu...",7,Transmit information or documents to customers...,5,None
2,Shopify MCP Server for Claude,Yes,"Plan, direct, and administer organizational fi...","Process orders, billing, and payments by calcu...",7,"Create, manage, or automate orders or invoices...",5;5;5;5;5,None


In [134]:
# View full details for a specific result
view_result(results_df, index=0)

Full Result for: SumoLogic MCP Server

Occupational Relevance: Yes

High-Level Task:
Plan, direct, and administer organizational financial, legal/regulatory, and records/IT operations by establishing policies and controls; collecting, verifying, analyzing, and publishing records, data, and documentary media; processing and reconciling transactions (pricing and quotes, orders/sales, billing/payroll/taxes, credit/loans, insurance, procurement/inventory, brokerage/trading, and customs duties); negotiating and managing contracts and agreements; conducting audits, inspections, background checks, and investigations (including fraud/forensic) and managing adjudicative, court, and case‑supervision activities; implementing security, access control, and information systems (document/content platforms, databases, and networks) to safeguard assets and ensure compliance; recruiting, training, evaluating, and administering compensation, benefits, and labor relations; representing the organization in

In [135]:
# Save results (uses SINGLE_MODEL_VERSION from config cell)
full_path, clean_path = save_results(
    results_df, 
    sample_df, 
    is_sample=True,
    version=SINGLE_MODEL_VERSION
)

Merged output shape: (3, 21)
Columns: ['title', 'url', 'key_features', 'uploaded_clean', 'text_for_llm', 'text_for_llm_2', 'len_text', 'row_idx', 'occupational_relevance', 'high_level_task_nums', 'high_level_task', 'mid_level_task_nums', 'mid_level_task', 'workflow_automation', 'onet_task_nums', 'onet_tasks', 'deployability', 'raw_response_1', 'raw_response_2', 'raw_response_3', 'error']
Results saved to: data/llm_classification/custom_tax_classification_sample_v5_20260130_112940.csv
Clean results saved to: data/llm_classification/custom_tax_classification_sample_v5_20260130_112940_clean.csv


---

## 3. Multi-Model Comparison

Run classification across multiple LLM providers to compare their outputs.

In [ ]:
# Configure models to compare
# MODELS_TO_RUN from lib/config.py contains all 8 models:
# ['claude-4.5-haiku', 'claude-sonnet-4', 'claude-sonnet-4.5', 'grok-3', 
#  'gpt-4.1', 'gpt-5', 'gpt-5-mini', 'gemini-2.5-pro']

# For a quick test, use a subset:
models_to_run = [
    "gpt-4.1",
    "claude-sonnet-4",
    # Uncomment to add more models:
    # "claude-sonnet-4.5",
    # "grok-3",
    # "gemini-2.5-pro",
]

# Or use all models from config:
# models_to_run = MODELS_TO_RUN

# Prepare sample
multi_sample_df = prepare_multi_model_sample(
    mcp_df,
    sample_titles={"Vapi MCP for Cursor", "Python Shopify MCP Server for Claude"}
)

In [ ]:
# Run multi-model classification (uses MULTI_MODEL_RUN_VERSION from config cell)
multi_results, output_folder = run_multi_model_classification(
    df=multi_sample_df,
    taxonomy=taxonomy,
    models_to_run=models_to_run,
    version=MULTI_MODEL_RUN_VERSION
)

print(f"\nResults stored for models: {list(multi_results.keys())}")
print(f"Output folder: {output_folder}")

In [ ]:
# Create side-by-side comparison
combined_df = create_combined_comparison(
    results_dict=multi_results,
    output_folder=output_folder,
    mcp_df_source=multi_sample_df
)

print(f"Combined comparison shape: {combined_df.shape}")
print(f"Columns: {list(combined_df.columns)}")

In [ ]:
# Build long-format comparison for analysis
long_df = build_long_model_comparison(output_folder, multi_sample_df)
long_df.to_csv(output_folder / "long_model_comparison.csv", index=False)
long_df.head()

---

## 4. Agreement Analysis

Analyze agreement between different models at each classification stage.

In [ ]:
# Analyze model agreement
agreement = analyze_model_agreement(
    results_dict=multi_results,
    output_folder=output_folder
)

In [ ]:
# View summary DataFrame
agreement['summary_df']

---

## 5. Determinism Check

Compare runs of the same model to verify consistency.

In [ ]:
# Compare two runs of the same model to check determinism
# Update these paths to actual result files from different runs

# Example usage:
# FILE_A = "data/llm_classification/multi_model_v1_20260129_180000/gpt-4.1_results.csv"
# FILE_B = "data/llm_classification/multi_model_v2_20260129_190000/gpt-4.1_results.csv"
# 
# determinism_results = compare_model_runs(
#     file_a=FILE_A,
#     file_b=FILE_B,
#     label_a="Run 1",
#     label_b="Run 2"
# )
#
# # View the comparison summary
# print(determinism_results['summary'])

print("To run determinism check, uncomment the code above and update file paths.")

---

## 6. Load Previous Results

Load and analyze results from a previous run.

In [ ]:
# Load and analyze results from a previous multi-model run
# Update path to your results folder

# Example usage:
# RESULTS_FOLDER = "data/llm_classification/multi_model_v1_20260129_180000"
# loaded_results, loaded_folder = load_multi_model_results(RESULTS_FOLDER)
# 
# # Analyze the loaded results
# analyze_model_agreement(loaded_results, loaded_folder)
# 
# # Create comparison table from loaded results
# long_df = build_long_model_comparison(loaded_folder, mcp_df)

print("To load previous results, uncomment the code above and update the folder path.")

---

## 7. Prompt Generator (Debugging)

Generate prompts for manual testing or debugging.

In [136]:
# Generate Stage 1 prompt
prompt_1 = generate_prompt_1(
    mcp_title="Vapi MCP for Cursor",
    mcp_df=mcp_df,
    taxonomy=taxonomy
)

if prompt_1:
    print("Stage 1 Prompt:")
    print("=" * 60)
    print(prompt_1[:2000] + "..." if len(prompt_1) > 2000 else prompt_1)

Stage 1 Prompt:
At the bottom of this prompt you will find a description and list of key features and use cases of an AI Model Context Protocol (MCP) server — a plugin-like system that allows AI assistants to access external tools, APIs, or data sources to perform real-world tasks.

You will answer two related questions about the MCP server. These questions are used together to determine whether the MCP performs occupationally relevant work and, if so, which branch of the O*NET taxonomy provided should be explored further to identify specific standardized tasks that may be automated or supported by the MCP.


Question 1: Occupational Relevance
<question>
Does this MCP server perform or significantly enable an occupationally relevant activity — that is, a concrete work activity that humans are commonly paid to perform within the formal economy and that aligns with standardized job tasks as represented in O*NET?
</question>

Follow these guidelines when answering Question 1:
- You MUST a

In [142]:
# Generate Stage 2 prompt (requires high-level task selection)
# Can provide multiple high-level tasks - they'll be combined for mid-level options
HIGH_LEVEL_SELECTED = [
    "Process orders, billing, and payments by calculating charges; creating orders or invoices; determining or verifying payment methods; receiving, recording, and issuing receipts for payments or deposits; authorizing returns, refunds, exchanges, or disbursements; and administering, maintaining, or developing purchasing, invoicing, or payment systems.|||Prepare, process, and manage business documents and records using computers and office equipment by composing and editing correspondence and reports; entering, formatting, and verifying data; preparing forms, invoices, and financial documents; assembling document packages; transmitting, filing, and retrieving records; and maintaining paper or electronic filing and database systems.",
    "Establish, maintain, and update manual or computerized records of transactions; inventories and inventory movement; shipments; orders, requisitions, or contracts; and customer accounts, interactions, and activity; track statuses, reconcile counts and amounts, and prepare routine reports."
]

prompt_2 = generate_prompt_2(
    mcp_title="Vapi MCP for Cursor",
    mcp_df=mcp_df,
    taxonomy=taxonomy,
    high_level_tasks=HIGH_LEVEL_SELECTED
)

if prompt_2:
    print("Stage 2 Prompt generated (length:", len(prompt_2), "chars)")

High-level task not found: Process orders, billing, and payments by calculati...


In [144]:
# Generate Stage 3 prompt (requires mid-level task selection)
# Can provide multiple mid-level tasks - they'll be combined for O*NET task options
MID_LEVEL_SELECTED = [
    "Process orders, billing, and payments by calculating charges; creating orders or invoices; determining or verifying payment methods; receiving, recording, and issuing receipts for payments or deposits; authorizing returns, refunds, exchanges, or disbursements; and administering, maintaining, or developing purchasing, invoicing, or payment systems.",
    "Prepare, process, and manage business documents and records using computers and office equipment by composing and editing correspondence and reports; entering, formatting, and verifying data; preparing forms, invoices, and financial documents; assembling document packages; transmitting, filing, and retrieving records; and maintaining paper or electronic filing and database systems.",
    "Establish, maintain, and update manual or computerized records of transactions; inventories and inventory movement; shipments; orders, requisitions, or contracts; and customer accounts, interactions, and activity; track statuses, reconcile counts and amounts, and prepare routine reports."
]

prompt_3 = generate_prompt_3(
    mcp_title="Vapi MCP for Cursor",
    mcp_df=mcp_df,
    taxonomy=taxonomy,
    mid_level_tasks=MID_LEVEL_SELECTED
)

if prompt_3:
    print("Stage 3 Prompt generated (length:", len(prompt_3), "chars)")
    # Optionally print the prompt
    print("=" * 60)
    # print(prompt_3[:2000] + "..." if len(prompt_3) > 2000 else prompt_3)
    print(prompt_3)

Stage 3 Prompt generated (length: 19460 chars)
At the bottom of this prompt you will find a description and list of key features and use cases of an AI Model Context Protocol (MCP) server — a plugin-like system that allows AI assistants to access external tools, APIs, or data sources to perform real-world tasks.

You will answer two related questions about the MCP server. These questions are used together to (1) determine which standardized occupational tasks from the O*NET database best represents the core work activity automated by the MCP, and (2) assess how deployable the MCP is for automating each task selected in real-world settings.

Follow these general guidelines when answering BOTH questions:
- Interpret MCPs on what they actually do or directly enable. Treat connected tools as part of the MCP's capability only if their functionality is directly accessible via the MCP's app or API.
- Interpret tasks focusing on action words and what the corresponding real-world human work act

---

## Notes

- **Sample Mode**: Filter `mcp_df` to a subset before running classification
- **Full Run**: Pass the entire `mcp_df` to `run_classification()` for all 8,953 MCPs
- **Checkpoints**: Results are saved periodically during classification
- **Concurrency**: Adjust via `max_concurrent` parameter (default: 50)
- **Output**: All results saved to `data/llm_classification/`